---
title: "Rotating drum: moving SDF walls (peclet.dem)"
subtitle: "Confine grains in a container written as a signed distance function, give the wall a velocity it never actually moves at, and watch a spinning drum carry the bed up its rising side into a steady cascade."
author: "Peclet"
date: "2026-07-05"
categories: [dem, sdf, walls, granular, rotating-drum, friction]
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/examples/rotating-drum/index.ipynb){target="_blank"}
&nbsp;Runs on a free Colab CPU runtime — the first cell installs `peclet` from PyPI.

## What you'll learn

In [custom-shaped particles](../sdf-particle-packing/index.qmd) the **particles** were
signed distance fields. Here the **container** is one too. `peclet.dem` lets you collide
grains against an arbitrary static **SDF wall** — a drum, a hopper, a vibrating tray — with
its own **binary particle–wall material**: a friction and restitution that belong to the
*pair* of surfaces, not to the grain alone. You will:

1. **Write a drum as an SDF** — a cylinder — and load it as a wall with its particle–wall
   friction and restitution.
2. Give that wall a **rigid-body velocity field** `v(x) = ω × (x − c)`. The geometry never
   moves, but a grain in contact feels the wall's velocity — exactly the trick a real
   rotating drum plays: the barrel spins, its shape does not change.
3. **Spin the drum** into the *rolling / cascading* regime and measure the **dynamic angle of
   repose** — wall friction drags grains up the rising side until the bed's free surface tilts
   to a steady angle and surface grains avalanche back down.

Everything runs on a CPU in about a minute.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (installs peclet from PyPI on Colab/Binder)"
import importlib.util, os, subprocess, sys
_local = os.environ.get("PECLET_LOCAL_BUILD")
if _local:
    for p in _local.split(os.pathsep):
        sys.path.insert(0, p)
elif importlib.util.find_spec("peclet") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peclet"], check=True)

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from matplotlib import animation
from peclet import dem
from peclet.dem import build_wall_sdf

plt.rcParams.update({"figure.dpi": 130, "font.size": 10, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
print("peclet.dem backend:", dem.execution_space)

## Step 1 — The drum is a signed distance function

Grains live in the **void** and the barrel is **solid**, so we adopt the natural wall
convention: the SDF is **positive where the grains are** and **negative inside the wall** —
the mirror image of a particle SDF. We study the drum **cross-section**: a cylinder of
radius `R` about the axis, made **periodic along its axis** (`z`) and only a couple of grains
thick, so it is effectively a 2-D disc — the clean roll you would see looking down the barrel.
That is a single signed distance, `R − r`:

In [ ]:
cx, cy = 12.0, 12.0        # drum axis (x, y), in grain radii
R      = 11.0              # barrel radius  (grain radius = 1 throughout)
Lz     = 2.5               # slab thickness along the (periodic) axis — one grain layer

def drum_sdf(p):
    """+ inside the void (where grains live), − inside the solid barrel."""
    return R - np.hypot(p[:, 0] - cx, p[:, 1] - cy)

::: {.callout-tip}
Working in **grain-radius units** (grain radius = 1, the default `global_scale`) keeps the
sphere–sphere contact model exact — size the *drum* in radii rather than shrinking the
grains. Any callable `f(points) → distance` works: swap `drum_sdf` for a hopper (a cone
`min`'d with a floor), a chute, or a boolean CSG of primitives.
:::

`build_wall_sdf` samples that callable onto a world-space lattice; `add_to` uploads it with
the **binary particle–wall material** — a grippy wall (`friction = 1.0`) so it can lift the
bed, and a low restitution so wall collisions are inelastic. The axis (`z`) is periodic so
grains recirculate rather than piling at an end.

In [ ]:
lo, hi = (0.0, 0.0, 0.0), (24.0, 24.0, Lz)
sim = dem.Simulation(600)
sim.set_sphere_shape(1.0)                        # grains of radius 1
sim.set_domain(lo, hi)
sim.enable_periodicity(False, False, True)       # periodic along the drum axis

wall = build_wall_sdf(drum_sdf, (lo, hi), resolution=110)
wid  = wall.add_to(sim, restitution=0.1, friction=1.0)   # <-- particle–wall material
print(f"wall SDF grid {wall.grid.shape}, range [{wall.grid.min():.1f}, {wall.grid.max():.1f}]")

Here is the drum's zero level set — the barrel the grains cannot cross:

In [ ]:
#| label: fig-sdf
#| fig-cap: "The drum as an SDF (cross-section). The black contour is the barrel wall; blue is the void the grains fill, red the solid wall the SDF pushes them out of."
gx = np.linspace(lo[0], hi[0], 240); gy = np.linspace(lo[1], hi[1], 240)
X, Y = np.meshgrid(gx, gy)
S = drum_sdf(np.stack([X.ravel(), Y.ravel(), np.zeros(X.size)], axis=1)).reshape(X.shape)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
im = ax.pcolormesh(X, Y, S, cmap="RdBu", vmin=-R, vmax=R, shading="auto")
ax.contour(X, Y, S, levels=[0], colors="k", linewidths=1.6)
ax.set_aspect("equal"); ax.set_title("drum SDF (cross-section)")
fig.colorbar(im, ax=ax, shrink=0.8, label="signed distance (+ void, − wall)")
plt.show()

## Step 2 — Fill the drum about half full

We drop grains in and pack them by **growth**: they start small (so nothing overlaps) and
grow to full size while the barrel confines them, gated on the measured overlap. A short
quench settles the bed. This is the same Lubachevsky–Stillinger protocol as the
[particle-packing example](../sdf-particle-packing/index.qmd) — but now the boundary is the
curved SDF wall, not six planes. We fill to a bit under half so there is a deep, flowing bed
(a single scattered layer just rolls; a bed several grains deep *cascades*).

In [ ]:
def fill_drum(friction_wall=1.0, friction_grain=0.8):
    sim = dem.Simulation(600)
    sim.set_sphere_shape(1.0)
    sim.set_domain(lo, hi)
    sim.enable_periodicity(False, False, True)
    wid = build_wall_sdf(drum_sdf, (lo, hi), resolution=110).add_to(
        sim, restitution=0.1, friction=friction_wall)
    sim.set_gravity(0.0, -20.0, 0.0)
    sim.set_material_params(0.1, 0.0, friction_grain)   # grain–grain: restitution, (unused), friction
    sim.set_solver_iterations(28, 8)

    g = np.arange(cx - R + 1.2, cx + R - 1.2, 2.15)
    pts = np.array([(x, y, Lz / 2) for x in g for y in g
                    if (x - cx)**2 + (y - cy)**2 < (R - 1.3)**2 and y < cy + 0.4 * R], np.float32)
    N = len(pts)
    p = np.zeros((N, 4), np.float32); p[:, :3] = pts; p[:, 3] = 1.0
    sim.set_positions(p); sim.set_scales(np.ones(N, np.float32)); sim.set_growth_params(1.0, 0.2)

    dt, crit = 0.004, 0.06
    for _ in range(1400):                         # grow, gated on overlap
        grow = sim.get_max_overlap() < crit and float(sim.get_scales().mean()) < 0.999
        sim.set_growth_params(1.0 if grow else 0.0, sim.get_growth_factor())
        sim.step(dt)
    sim.set_growth_params(0.0, sim.get_growth_factor())
    for _ in range(900):                          # settle
        sim.step(dt)
    return sim, wid, N

dt = 0.004
t0 = time.time()
sim, wid, N = fill_drum()
print(f"packed {N} grains in {time.time()-t0:.1f} s; max overlap = {sim.get_max_overlap():.3f} (grain r = 1)")

## Step 3 — Spin the wall

The drum's surface velocity is a **rigid-body field** `v(x) = v_lin + ω × (x − center)`. For
rotation we set the angular velocity about the axis point `(cx, cy)`; grains that touch the
barrel feel that tangential velocity through **wall friction** and are carried up the rising
side. `set_wall_velocity` is cheap (a few scalars) — set it once for a steady drum, or every
step for a vibrating wall (see *Adapt this*). We keep the [Froude
number](https://en.wikipedia.org/wiki/Froude_number) `ω²R/g ≈ 0.45` in the *cascading* regime,
where the bed rolls rather than centrifuging onto the wall.

In [ ]:
omega = 0.9     # rad/s, counter-clockwise about +z  (Froude number ω²R/g ≈ 0.45)
pre_spin = sim.get_positions().reshape(-1, 3).copy()     # the bed at rest, for the angle baseline
sim.set_wall_velocity(wid, lin_vel=(0, 0, 0), ang_vel=(0, 0, omega), center=(cx, cy, 0))

frames = []
def snapshot():
    pos = sim.get_positions().reshape(-1, 3)
    vel = sim.get_velocities().reshape(-1, 3)
    frames.append((pos[:, 0].copy(), pos[:, 1].copy(), np.linalg.norm(vel, axis=1)))

for f in range(90):                               # ~ a couple of revolutions
    for _ in range(25):
        sim.step(dt)
    snapshot()

pos = sim.get_positions().reshape(-1, 3)
esc = int((np.hypot(pos[:, 0] - cx, pos[:, 1] - cy) > R + 1.0).sum())
peak = max(f[2].max() for f in frames[30:])
print(f"grains that escaped the barrel: {esc} / {N}")
print(f"peak grain speed = {peak:.1f}  vs wall surface speed ωR = {omega*R:.1f} "
      f"(grains never outrun the wall — no spurious energy)")

## The cascading bed

Coloured by speed, the spinning bed is unmistakable: it climbs the rising (`+x`) wall, shears
along a tilted free surface, and avalanches back down — the grains near the barrel move
fastest (they share the wall's speed), those in the passive core barely at all.

In [ ]:
#| label: fig-drum-anim
#| fig-cap: "The rotating drum (counter-clockwise), grains coloured by speed. Wall friction drags the bed up the rising side to a steady **dynamic angle of repose**, where surface grains avalanche back — the signature of a moving SDF wall."
theta = np.linspace(0, 2*np.pi, 200)
vmax = max(f[2].max() for f in frames) * 0.75
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.plot(cx + R*np.cos(theta), cy + R*np.sin(theta), "k-", lw=2)   # barrel
scat = ax.scatter(frames[0][0], frames[0][1], c=frames[0][2], s=60,
                  cmap="viridis", vmin=0, vmax=vmax, edgecolors="0.2", linewidths=0.3)
ax.annotate("ω", xy=(cx - 0.80*R, cy + 0.64*R), fontsize=13, color="0.35")
ax.annotate("", xy=(cx - 0.88*R, cy + 0.38*R), xytext=(cx - 0.62*R, cy + 0.68*R),
            arrowprops=dict(arrowstyle="->", color="0.5", lw=1.6, connectionstyle="arc3,rad=0.3"))
ax.set_xlim(cx - R - 2, cx + R + 2); ax.set_ylim(cy - R - 2, cy + R + 2)
ax.set_aspect("equal"); ax.axis("off"); ax.set_title("rotating drum — CCW")

def update(i):
    x, y, s = frames[i]
    scat.set_offsets(np.column_stack([x, y])); scat.set_array(s)
    return scat,

anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=80, blit=True)
plt.close(fig)
from IPython.display import HTML
HTML(anim.to_jshtml())

## The dynamic angle of repose

The headline: the tilt of the bed's **free surface**. We fit a line to the top grain in each
vertical strip across the central span of the drum (away from where the bed climbs the wall) and
read its slope. A rotating bed does not tilt to a *fixed* angle — it **avalanches
intermittently**, so the surface angle oscillates about a steady mean. We therefore track it over
the whole spin: it starts flat (the bed at rest), rises as friction drags the bed up, and then
fluctuates about the dynamic angle of repose.

In [ ]:
#| label: fig-repose
#| fig-cap: "Left: the free-surface angle over the spin — flat at rest, then a sawtooth as the surface is dragged up to its maximum angle of repose and released by a bulk avalanche (the slumping regime). Right: the bed at the last frame with the fitted free surface (red)."
def surface_angle(x, y, band=0.62):
    """Slope angle (deg) and fitted line of the top-of-bed surface across the drum's central band."""
    xc, yc = x - cx, y - cy
    keep = np.abs(xc) < band * R
    xc, yc = xc[keep], yc[keep]
    bins = np.linspace(xc.min(), xc.max(), 9)
    idx = np.clip(np.digitize(xc, bins), 1, len(bins) - 1)
    xs, ys = [], []
    for b in range(1, len(bins)):
        m = idx == b
        if m.any():
            xs.append(xc[m].mean()); ys.append(yc[m].max())
    xs, ys = np.array(xs), np.array(ys)
    slope, intercept = np.polyfit(xs, ys, 1)
    return np.degrees(np.arctan(slope)), (slope, intercept)

t = np.arange(len(frames)) * 25 * dt
ang_t = np.array([surface_angle(fx, fy)[0] for fx, fy, _ in frames])
ang_stat = surface_angle(pre_spin[:, 0], pre_spin[:, 1])[0]      # same bed, at rest
steady = ang_t[len(ang_t) // 2:]                                # second half = steady regime
ang_dyn, ang_sd = steady.mean(), steady.std()
ang_peak = steady.max()                                         # avalanche-triggering angle

fig, (axL, axR) = plt.subplots(1, 2, figsize=(9.0, 4.2))
axL.axhline(ang_stat, color="0.5", ls="--", lw=1.2, label=f"at rest ({ang_stat:+.0f}°)")
axL.axhline(ang_dyn, color="crimson", lw=1.5, label=f"dynamic mean ({ang_dyn:+.0f}°)")
axL.fill_between(t, ang_dyn - ang_sd, ang_dyn + ang_sd, color="crimson", alpha=0.12)
axL.plot(t, ang_t, color="#2a6fdb", lw=1.2)
axL.set_xlabel("time"); axL.set_ylabel("free-surface angle (°)"); axL.legend(loc="lower right")
axL.set_title("build up to the repose angle, then avalanche"); axL.grid(True, alpha=0.3)

fx, fy, fs = frames[-1]
_, (slope, intercept) = surface_angle(fx, fy)
axR.plot(cx + R*np.cos(theta), cy + R*np.sin(theta), "k-", lw=1.5)
axR.scatter(fx, fy, s=55, c=fs, cmap="viridis", vmin=0, vmax=vmax, edgecolors="0.3", linewidths=0.3)
xx = np.array([-0.62 * R, 0.62 * R])
axR.plot(cx + xx, cy + slope * xx + intercept, "-", color="crimson", lw=3)
axR.set_aspect("equal"); axR.axis("off"); axR.set_title(f"last frame:  {surface_angle(fx, fy)[0]:+.0f}°")
plt.show()
print(f"at rest:  surface angle ≈ {ang_stat:+.1f}°  (a settled bed, near flat)")
print(f"spinning: surface builds to a maximum angle of repose ≈ {ang_peak:+.1f}° then avalanches back")
print(f"          (mean {ang_dyn:+.1f}° ± {ang_sd:.0f}° over the avalanche cycle)")

At rest the surface is flat. Spinning, this small bead bed sits in the **intermittent-avalanche
(slumping) regime**: wall friction drags the surface up until it reaches its **maximum angle of
repose**, a bulk avalanche releases it back toward flat, and the cycle repeats — the sawtooth in the
plot. That periodic build-and-release is real granular physics, not solver noise (grains never
outrun the wall — the peak-speed check above is how we know the contact solve is not injecting
energy). The angle is gentle because smooth spheres *roll*: a bead bed cascades far shallower than
angular sand (~25–35°). Grip the grains harder (raise `friction`), fill deeper, spin faster (a higher
Froude number moves into the *continuous cascading* regime), or use non-spherical
[`build_particle`](../sdf-particle-packing/index.qmd) shapes, and the angle steepens.

::: {.callout-note}
## What "wrong" looks like — a debugging note
An earlier version of the moving-wall contact solve **injected energy** on the downhill side of the
cascade: grains that should have rolled gently back down instead *jumped*, and their peak speed shot
past the wall speed. The tell was quantitative — `peak grain speed` above `ωR` is unphysical for a
driven-by-friction bed. The [DEM sanity checks](../../sanity-checks/index.qmd) pin down exactly this
kind of regression; the fix restored `peak speed < ωR`, printed above.
:::

## Adapt this yourself

- **Vibrate the wall instead of spinning it.** A moving wall needs no rotation — drive the
  *linear* velocity sinusoidally each step to shake the bed (the geometry stays put):
  ```python
  for k in range(nsteps):
      v = A * omega_v * np.cos(omega_v * k * dt)      # oscillating wall velocity
      sim.set_wall_velocity(wid, lin_vel=(0, v, 0), ang_vel=(0, 0, 0), center=(cx, cy, 0))
      sim.step(dt)
  ```
- **Change the material.** `friction` and `restitution` in `add_to` are the *binary*
  particle–wall pair — a slick wall (`friction≈0.1`) lets the bed slip and slump; a grippy
  wall lifts it further and steepens the angle. They are independent of the grain–grain
  `set_material_params`.
- **Change the container.** Any SDF works: a **hopper** is a cone `min`'d with a floor; a
  **chute** is a tilted slab; add lifters/baffles by `min`-ing more primitives. Keep the sign
  convention (**+ in the void, − in the wall**) and cover the whole domain with the grid.
- **Non-spherical grains.** Swap `set_sphere_shape` for an imported
  [`build_particle`](../sdf-particle-packing/index.qmd) shape — the same wall collides against
  each surface point of the grain, so rods and beans tumble (and cascade at a steeper angle).
- **Go bigger / on a GPU or cluster.** Increase the grain count and drum size; on a CUDA/HIP
  build the identical script runs on the device, and a distributed build adds
  `sim.init_mpi(...)` / `sim.step_mpi(...)` (use a closed or rounded drum under MPI).

## Reproduce this

```bash
# from a checkout of peclet-examples
pip install -e '.[sim]'          # or: pip install peclet
# refresh the rendered page from a local build of the suite:
PECLET_LOCAL_BUILD=/path/to/suite/dem/build_omp \
  quarto render examples/rotating-drum/index.qmd --execute
```